# Thermodynamically Consistent Lattice ABP MIPS Demo

This notebook runs the `data.lattice_abp_tc` implementation of the Kim-Kwon-Baek thermodynamically consistent lattice Monte Carlo method. It saves true medium entropy production from accepted hops and provides simple diagnostics for MIPS-like density inhomogeneity.

For a quick check, reduce `n_steps`. For clearer MIPS, increase `n_steps`, `L`, and `grid_size` while keeping `prefactor="cv"`.

In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "data" / "lattice_abp_tc" / "core.py").exists():
            return path
    raise RuntimeError("Could not find CNEEP_v2 repo root from current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data.lattice_abp_tc import ThermodynamicLatticeABP, ThermodynamicLatticeABPParams

ROOT

In [ ]:
def n_from_packing_fraction(phi, box_size, sigma):
    return int(round(phi * 4.0 * box_size * box_size / (math.pi * sigma * sigma)))


# MIPS-friendly defaults. Use smaller n_steps for a smoke run.
L = 24.0
grid_size = 96
sigma = 1.0
phi = 0.55
N = n_from_packing_fraction(phi, L, sigma)

params = ThermodynamicLatticeABPParams(
    N=N,
    L=L,
    grid_size=grid_size,
    sigma=sigma,
    epsilon=20.0,
    v0=24.0,
    Dr=1.0,
    Dt=0.2,
    dt=1.0e-4,
    prefactor="cv",
    seed=7,
    device="auto",
)

n_steps = 5000
burn_in = 0
save_interval = 100
coarse_box = 12
B = 1

sim = ThermodynamicLatticeABP(params)
print(
    f"device={sim.device}, L={params.L:g}, grid={params.grid_size}, dl={params.dl:g}, "
    f"N={params.N}, phi={params.phi:.3f}, Pe={params.Pe:.2f}, "
    f"gamma2={params.gamma2:.3g}, prefactor={params.prefactor}"
)

In [ ]:
result = sim.simulate(
    B=B,
    burn_in=burn_in,
    n_steps=n_steps,
    save_interval=save_interval,
    show_progress=True,
    save_occupancy=True,
    save_exact_medium_ep=True,
)

print("sites", tuple(result["sites"].shape))
print("occupancy", tuple(result["occupancy"].shape))
print("exact_medium_ep", tuple(result["exact_medium_ep"].shape))

In [ ]:
initial_summary = sim.mips_summary_from_sites(result["sites"][0], coarse_box=coarse_box)
final_summary = sim.mips_summary_from_sites(result["sites"][-1], coarse_box=coarse_box)

summary = {
    "initial": initial_summary,
    "final": final_summary,
    "mean_exact_medium_ep_rate_by_ensemble": result["exact_medium_ep_rate"].mean(dim=1).numpy().tolist(),
}

print(json.dumps(summary, indent=2))

In [ ]:
occ0 = result["occupancy"][0, 0].numpy()
occ1 = result["occupancy"][-1, 0].numpy()
coarse0 = sim.coarse_density(torch.as_tensor(occ0), box=coarse_box)[0].cpu().numpy()
coarse1 = sim.coarse_density(torch.as_tensor(occ1), box=coarse_box)[0].cpu().numpy()

fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
panels = [
    (occ0, "initial occupancy", "gray_r"),
    (occ1, "final occupancy", "gray_r"),
    (coarse0, "initial coarse density", "viridis"),
    (coarse1, "final coarse density", "viridis"),
]
for ax, (image, title, cmap) in zip(axes.ravel(), panels):
    im = ax.imshow(image.T, origin="lower", interpolation="nearest", cmap=cmap)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()

In [ ]:
dt_saved = save_interval * params.dt
times_mid = result["times"][:-1].numpy() + 0.5 * dt_saved
ep_rate = result["exact_medium_ep_rate"][0].numpy()
active_rate = result["exact_active_medium_ep"][0].numpy() / dt_saved
wca_rate = result["exact_wca_medium_ep"][0].numpy() / dt_saved

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times_mid, ep_rate, label="total medium EP rate")
ax.plot(times_mid, active_rate, label="active contribution", alpha=0.8)
ax.plot(times_mid, wca_rate, label="WCA boundary contribution", alpha=0.8)
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("time")
ax.set_ylabel("EP rate")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
outdir = ROOT / "output" / "lattice_abp_tc_notebook"
outdir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    outdir / "trajectory.npz",
    sites=result["sites"].numpy(),
    occupancy=result["occupancy"].numpy(),
    theta=result["theta"].numpy(),
    times=result["times"].numpy(),
    exact_medium_ep=result["exact_medium_ep"].numpy(),
    exact_medium_ep_rate=result["exact_medium_ep_rate"].numpy(),
)

with open(outdir / "summary.json", "w", encoding="utf-8") as f:
    json.dump({"params": params.__dict__, **summary}, f, indent=2)

print(f"Saved outputs under {outdir}")